# garcon SFT: SmolLM2-135M-Instruct on Korean NL2BASH

Korean natural language → Linux command intent classification.

- **Model**: `HuggingFaceTB/SmolLM2-135M-Instruct`
- **Dataset**: `rlawltjd/korean-nl2bash` (8,089 pairs)
- **Method**: SFT + LoRA (PEFT)
- **Format**: OpenAI ChatML (Thought / Action / Action Input)

Prerequisites:
```bash
pip install transformers datasets peft trl bitsandbytes accelerate
```

## 1. Setup

In [ ]:
import os
import json
import re
from pathlib import Path

import torch
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

os.environ["TOKENIZERS_PARALLELISM"] = "false"

MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
DS_ID = "rlawltjd/korean-nl2bash"
OUTPUT_DIR = Path("fine_tuning/output/sft_nl2bash_0135m_lora")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 2. Load Model (NF4 Quantization for QLoRA)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if DEVICE != "cpu" else torch.float32,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
# SmolLM2 uses special chat template — preserve it
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto" if DEVICE in ("cuda", "mps") else None,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16 if DEVICE != "cpu" else torch.float32,
)
model.config.use_cache = False

print(f"Model loaded: {MODEL_ID}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

## 3. Load & Convert NL2BASH Dataset → Garcon ChatML Format

In [ ]:
SYSTEM_PROMPT = """You are garcon, a Korean-language terminal assistant.\nYou help users by running Linux commands step by step.\n\nYou have access to the following tools:\n[\n  {\"name\": \"ls_command\", \"parameters\": {\"type\": \"object\", \"properties\": {\"path\": {\"type\": \"string\"}, \"options\": {\"type\": \"string\"}}, \"required\": [\"path\"]}},\n  {\"name\": \"find_command\", \"parameters\": {\"type\": \"object\", \"properties\": {\"path\": {\"type\": \"string\"}, \"name\": {\"type\": \"string\"}}, \"required\": [\"path\"]}},\n  {\"name\": \"grep_command\", \"parameters\": {\"type\": \"object\", \"properties\": {\"pattern\": {\"type\": \"string\"}, \"path\": {\"type\": \"string\"}}, \"required\": [\"pattern\", \"path\"]}},\n  {\"name\": \"cat_command\", \"parameters\": {\"type\": \"object\", \"properties\": {\"path\": {\"type\": \"string\"}}, \"required\": [\"path\"]}},\n  {\"name\": \"wc_command\", \"parameters\": {\"type\": \"object\", \"properties\": {\"path\": {\"type\": \"string\"}, \"options\": {\"type\": \"string\"}}, \"required\": [\"path\"]}},\n  {\"name\": \"head_command\", \"parameters\": {\"type\": \"object\", \"properties\": {\"path\": {\"type\": \"string\"}, \"lines\": {\"type\": \"integer\"}}, \"required\": [\"path\"]}},\n  {\"name\": \"tail_command\", \"parameters\": {\"type\": \"object\", \"properties\": {\"path\": {\"type\": \"string\"}, \"lines\": {\"type\": \"integer\"}}, \"required\": [\"path\"]}},\n  {\"name\": \"mkdir_command\", \"parameters\": {\"type\": \"object\", \"properties\": {\"path\": {\"type\": \"string\"}}, \"required\": [\"path\"]}},\n  {\"name\": \"rm_command\", \"parameters\": {\"type\": \"object\", \"properties\": {\"path\": {\"type\": \"string\"}, \"recursive\": {\"type\": \"boolean\"}}, \"required\": [\"path\"]}},\n  {\"name\": \"cp_command\", \"parameters\": {\"type\": \"object\", \"properties\": {\"source\": {\"type\": \"string\"}, \"destination\": {\"type\": \"string\"}}, \"required\": [\"source\", \"destination\"]}},\n  {\"name\": \"mv_command\", \"parameters\": {\"type\": \"object\", \"properties\": {\"source\": {\"type\": \"string\"}, \"destination\": {\"type\": \"string\"}}, \"required\": [\"source\", \"destination\"]}},\n  {\"name\": \"chmod_command\", \"parameters\": {\"type\": \"object\", \"properties\": {\"mode\": {\"type\": \"string\"}, \"path\": {\"type\": \"string\"}}, \"required\": [\"mode\", \"path\"]}},\n  {\"name\": \"tar_command\", \"parameters\": {\"type\": \"object\", \"properties\": {\"operation\": {\"type\": \"string\"}, \"archive\": {\"type\": \"string\"}, \"files\": {\"type\": \"string\"}}, \"required\": [\"operation\", \"archive\"]}},\n  {\"name\": \"sort_command\", \"parameters\": {\"type\": \"object\", \"properties\": {\"path\": {\"type\": \"string\"}}, \"required\": [\"path\"]}},\n  {\"name\": \"uniq_command\", \"parameters\": {\"type\": \"object\", \"properties\": {\"path\": {\"type\": \"string\"}}, \"required\": [\"path\"]}},\n  {\"name\": \"diff_command\", \"parameters\": {\"type\": \"object\", \"properties\": {\"path1\": {\"type\": \"string\"}, \"path2\": {\"type\": \"string\"}}, \"required\": [\"path1\", \"path2\"]}},\n  {\"name\": \"tree_command\", \"parameters\": {\"type\": \"object\", \"properties\": {\"path\": {\"type\": \"string\"}}, \"required\": [\"path\"]}},\n  {\"name\": \"Finish\", \"parameters\": {\"type\": \"object\", \"properties\": {\"return_type\": {\"type\": \"string\"}, \"final_answer\": {\"type\": \"string\"}}, \"required\": [\"return_type\"]}}\n]\n\nAlways respond in the following format:\nThought: (your reasoning in Korean)\nAction: (command name from the list)\nAction Input: (JSON object with parameters)"""

VALID_ACTIONS = {
    "ls_command", "find_command", "grep_command", "cat_command",
    "wc_command", "head_command", "tail_command", "mkdir_command",
    "rm_command", "cp_command", "mv_command", "chmod_command",
    "tar_command", "sort_command", "uniq_command", "diff_command",
    "tree_command", "Finish",
}

# ---- heuristic command extractor ----
def extract_command_from_bash(bash_cmd: str, instruction: str) -> tuple[str, dict]:
    """Heuristically map a bash command to garcon's command set."""
    cmd = bash_cmd.strip()
    tokens = cmd.split()
    base = tokens[0] if tokens else ""

    # ls variants
    if base in ("ls", "ll", "dir"):
        path = "."
        options = "-la" if base == "ll" else ""
        for tok in tokens[1:]:
            if tok.startswith("-"):
                options = (options + tok).replace("-la", "-la").replace("-al", "-la")
            elif tok not in ("-la", "-al", "-lah"):
                path = tok
        return "ls_command", {"path": path, "options": options or "-la"}

    # cat / head / tail
    if base == "cat":
        return "cat_command", {"path": tokens[-1] if len(tokens) > 1 else "."}
    if base == "head":
        lines = 10
        path = tokens[-1]
        if "-n" in tokens:
            try:
                lines = int(tokens[tokens.index("-n") + 1])
            except (IndexError, ValueError):
                pass
        return "head_command", {"path": path, "lines": lines}
    if base == "tail":
        lines = 10
        path = tokens[-1]
        if "-n" in tokens:
            try:
                lines = int(tokens[tokens.index("-n") + 1])
            except (IndexError, ValueError):
                pass
        return "tail_command", {"path": path, "lines": lines}

    # wc
    if base == "wc":
        options = "-l" if "-l" in tokens else ""
        path = tokens[-1] if len(tokens) > 1 else "."
        return "wc_command", {"path": path, "options": options}
    if base == "grep" or base == "egrep":
        pattern = tokens[1].strip('"\'') if len(tokens) > 1 else ""
        path = "."
        for tok in tokens[2:]:
            if not tok.startswith("-"):
                path = tok
                break
        if "-r" in tokens:
            path = "."  # grep -r means recursive from .
        return "grep_command", {"pattern": pattern, "path": path}

    # find
    if base == "find":
        path = tokens[1] if len(tokens) > 1 else "."
        name = ""
        if "-name" in tokens:
            try:
                name = tokens[tokens.index("-name") + 1].strip('"\'')
            except IndexError:
                pass
        return "find_command", {"path": path, "name": name}

    # mkdir
    if base == "mkdir":
        return "mkdir_command", {"path": tokens[-1] if len(tokens) > 1 else ""}

    # cp / mv
    if base == "cp" and "-r" not in tokens:
        return "cp_command", {"source": tokens[-2], "destination": tokens[-1]}
    if base == "mv":
        if len(tokens) >= 3:
            return "mv_command", {"source": tokens[-2], "destination": tokens[-1]}

    # rm
    if base == "rm":
        rec = "-r" in tokens or "-rf" in tokens
        targets = [t for t in tokens[1:] if not t.startswith("-")]
        return "rm_command", {"path": targets[0] if targets else "", "recursive": rec}

    # chmod
    if base == "chmod":
        return "chmod_command", {"mode": tokens[1], "path": tokens[2] if len(tokens) > 2 else ""}

    # tar
    if base == "tar":
        if "-x" in cmd or "--extract" in cmd:
            archive = ""
            for i, tok in enumerate(tokens):
                if tok in ("-f", "--file") and i + 1 < len(tokens):
                    archive = tokens[i + 1]
            return "tar_command", {"operation": "extract", "archive": archive, "files": ""}
        elif "-c" in cmd or "--create" in cmd:
            archive = ""
            files = ""
            for i, tok in enumerate(tokens):
                if tok in ("-f", "--file") and i + 1 < len(tokens):
                    archive = tokens[i + 1]
            files = tokens[-1] if tokens[-1] != archive else ""
            return "tar_command", {"operation": "compress", "archive": archive, "files": files}
        return "tar_command", {"operation": "list", "archive": tokens[-1], "files": ""}

    # sort / uniq / diff
    if base == "sort":
        return "sort_command", {"path": tokens[-1] if len(tokens) > 1 else "."}
    if base == "uniq":
        return "uniq_command", {"path": tokens[-1] if len(tokens) > 1 else "."}
    if base == "diff":
        return "diff_command", {"path1": tokens[-2], "path2": tokens[-1]}

    # refuse pattern
    if base in ("sudo", "su", "passwd", "kill", "pkill", "reboot"):
        return "refuse_command", {}

    # Default: Finish with answer
    return "Finish", {"return_type": "final", "final_answer": cmd}

def to_chatml(example: dict) -> dict:
    instruction = example["instruction"]
    bash = example["output"]
    action, params = extract_command_from_bash(bash, instruction)

    if action not in VALID_ACTIONS:
        action = "Finish"
        params = {"return_type": "final", "final_answer": bash}

    thought_map = {
        "ls_command": "디렉토리 내용을 확인합니다.",
        "cat_command": "파일 내용을 읽습니다.",
        "head_command": "파일의 처음 부분을 확인합니다.",
        "tail_command": "파일의 마지막 부분을 확인합니다.",
        "wc_command": "파일의 통계를 확인합니다.",
        "grep_command": "지정된 패턴으로 검색합니다.",
        "find_command": "파일을 검색합니다.",
        "mkdir_command": "새 디렉토리를 생성합니다.",
        "cp_command": "파일을 복사합니다.",
        "mv_command": "파일을 이동하거나 이름을 변경합니다.",
        "rm_command": "파일을 삭제합니다.",
        "chmod_command": "파일 권한을 변경합니다.",
        "tar_command": "압축 파일을 처리합니다.",
        "sort_command": "파일 내용을 정렬합니다.",
        "uniq_command": "중복 라인을 제거합니다.",
        "diff_command": "두 파일을 비교합니다.",
        "tree_command": "디렉토리 구조를 확인합니다.",
        "Finish": "작업을 완료합니다.",
    }
    thought = thought_map.get(action, "요청을 처리합니다.")

    assistant = f"Thought: {thought}\nAction: {action}\nAction Input: {json.dumps(params, ensure_ascii=False)}"

    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": instruction},
            {"role": "assistant", "content": assistant},
        ]
    }

# Load and convert
raw_ds = load_dataset(DS_ID)
converted = raw_ds["train"].map(to_chatml, remove_columns=raw_ds["train"].column_names)

# Train/valid split
split = converted.train_test_split(test_size=0.1, seed=42)
dataset = DatasetDict({
    "train": split["train"],
    "validation": split["test"],
})

print(f"Train: {len(dataset['train'])}, Valid: {len(dataset['validation'])}")
print("\n--- Sample ---")
print(json.dumps(dataset['train'][0], indent=2, ensure_ascii=False))

## 4. LoRA Configuration

In [ ]:
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                    # rank
    lora_alpha=32,           # scaling
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    use_rslora=False,        # standard LoRA (rslora for very small r)
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 5. Training

In [ ]:
training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    max_seq_length=1024,
    bf16=DEVICE in ("cuda", "mps"),
    fp16=False,
    optim="paged_adamw_8bit",
    group_by_length=True,
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    args=training_args,
)

trainer.train()

## 6. Save Adapter & Push to Hub (optional)

In [ ]:
# Save LoRA adapter locally
adapter_dir = OUTPUT_DIR / "final_adapter"
trainer.save_model(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print(f"Adapter saved to: {adapter_dir}")

# Optional: merge and save full model (no adapter needed)
# merged = model.merge_and_unload()
# merged.save_pretrained(str(OUTPUT_DIR / "merged"))
# tokenizer.save_pretrained(str(OUTPUT_DIR / "merged"))

# Optional: push to HuggingFace Hub
# from huggingface_hub import login
# login()  # enter your token
# trainer.push_to_hub("your-handle/garcon-smollm2-135m-nl2bash-lora")

## 7. Inference Test

In [ ]:
def generate(instruction: str, max_new_tokens: int = 128) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": instruction},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.1,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    decoded = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
    return decoded

test_cases = [
    "현재 폴더 파일 목록 보여줘",
    "downloads 폴더에서 py 파일 찾아줘",
    "README.md 내용 읽어줘",
    "test.py 몇 줄이야?",
    "로그에서 에러 찾아줘",
    "새로운 폴더 만들어줘 temp",
]

for tc in test_cases:
    print(f"\n👤 {tc}")
    print(f"🤖 {generate(tc)}")

## 8. Export to GGUF (for llama.cpp runtime)

In [ ]:
# Optional: export merged model to GGUF for llama.cpp inference in garcon
# Requires llama.cpp's convert_hf_to_gguf.py or unsloth's export

# from unsloth import export_to_gguf  # if using unsloth
# export_to_gguf(
#     model, tokenizer,
#     quantization_method="q4_k_m",
#     output_file=str(OUTPUT_DIR / "garcon-smollm2-135m-q4_k_m.gguf")
# )

# Or manually with llama.cpp:
# git clone https://github.com/ggerganov/llama.cpp
# python llama.cpp/convert_hf_to_gguf.py fine_tuning/output/sft_nl2bash_0135m_lora/merged/ \
#     --outfile fine_tuning/output/garcon-smollm2-135m-q4_k_m.gguf \
#     --outtype q4_k_m

## Notes

- **Heuristic mapping**: NL2BASH outputs raw bash commands. The `extract_command_from_bash()` function maps them to garcon's command vocabulary. Not all bash commands map cleanly — edge cases default to `Finish`.
- **Fine-tuning targets**: The model learns the Thought/Action/Action Input format. After training, garcon's `model_router.py` should parse the output identically (same regex/grammar).
- **Data quality**: NL2BASH contains English + Korean mixed instructions. For production, supplement with Korean-only synthetic data (`scripts/generate_training_data.py`).
- **Hardware**: 135M model + LoRA fine-tuning fits on any GPU with 8GB VRAM (Colab T4 works). CPU training is possible but slow.
